# Publish reviewed model artifacts for the dashboard
Run from the gold_forecasting project after notebook 01 and model notebooks. This notebook copies existing saved bundles and forecasts; it never trains. Choose a new immutable version before publishing. Promotion is explicit. Existing five-horizon results are forecast-only, so do not manufacture a test report.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from model_lib.preprocessing import ROOT, load_prepared
from model_lib.artifacts.registry import ArtifactStore
from model_lib.reporting.exports import publish_saved_model, publish_future_table
load_prepared()  # Verify configuration, raw sources, and aligned features first.
store = ArtifactStore(Path(os.environ.get('GOLD_ARTIFACT_ROOT', ROOT / 'artifacts/serving')))
VERSION = 'initial-2026-10-26'
MODEL_IDS = ['prophet', 'xgboost', 'gru', 'lstm']

## Publish immutable versions and matching snapshots
Rerunning with an existing version intentionally raises FileExistsError. Use a new version for changed artifacts. The snapshot records auxiliary Prophet paths separately from live spot inference's origin-held scenarios.

In [ ]:
for model_id in MODEL_IDS:
    metadata = publish_saved_model(ROOT, store, model_id, VERSION)
    forecast = pd.read_csv(ROOT / 'artifacts' / model_id / 'future_forecast.csv')
    assumptions = ('Notebook-generated anchored auxiliary regressor paths.' if model_id == 'prophet' else 'Origin-time model features; no realized future external values.')
    publish_future_table(store, metadata, VERSION, forecast, assumptions=assumptions)
    display(metadata.model_dump())

In [ ]:
# Run after reviewing the publications above. Rollback uses this same method with an earlier version.
for model_id in MODEL_IDS:
    store.promote(model_id, VERSION)
display(store.active_versions())

## Publish holdout results only after actual notebook evaluation
Use `model_lib.reporting.exports.publish_test_table(store, evaluation_metadata, report_id, holdout_predictions, assumptions=...)`. `evaluation_metadata` must describe the model fitted before the test origins, not a later final-refit bundle. This helper saves residuals and metrics during notebook execution. Current forecast-only outputs do not qualify.